# Solutions — Datetime Functions Basics

**Dataset:** `samples.nyctaxi.trips`, `samples.bakehouse.sales_transactions`, `samples.wanderbricks.bookings`

**Topics:** year, month, hour, datediff, date_format, date_trunc, when/otherwise

> Reference solutions for practice notebooks.
> **Try the problem yourself first** — only open this when you've made a genuine attempt.
>
> Each solution includes a `# Why:` comment explaining the approach chosen.

In [ ]:
from pyspark.sql import functions as F, types as T
trips        = spark.table("samples.nyctaxi.trips")
transactions = spark.table("samples.bakehouse.sales_transactions")
bookings     = spark.table("samples.wanderbricks.bookings")

---
## Problem 1 — Extract Date Components

In [ ]:
# Why: all date functions are applied in one select — no intermediate columns needed.
result_1 = trips.select(
    "tpep_pickup_datetime",
    F.year("tpep_pickup_datetime").alias("year"),
    F.month("tpep_pickup_datetime").alias("month"),
    F.dayofmonth("tpep_pickup_datetime").alias("day"),
    F.hour("tpep_pickup_datetime").alias("hour"),
    F.minute("tpep_pickup_datetime").alias("minute")
)

---
## Problem 2 — Calculate Trip Duration

In [ ]:
# Why: cast to long (Unix epoch seconds) to enable arithmetic; divide by 60 for minutes.
result_2 = (
    trips.withColumn(
        "duration_minutes",
        F.round(
            (F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60,
            1
        )
    ).filter(F.col("duration_minutes") > 0)
     .select("tpep_pickup_datetime", "tpep_dropoff_datetime", "duration_minutes")
)

---
## Problem 3 — Day of Week Analysis

In [ ]:
# Why: when().when()...otherwise() maps numeric codes to readable names.
# dayofweek: 1=Sunday, 2=Monday, ..., 7=Saturday in Spark.
result_3 = (
    transactions.withColumn("day_name",
        F.when(F.dayofweek("dateTime") == 1, "Sunday")
         .when(F.dayofweek("dateTime") == 2, "Monday")
         .when(F.dayofweek("dateTime") == 3, "Tuesday")
         .when(F.dayofweek("dateTime") == 4, "Wednesday")
         .when(F.dayofweek("dateTime") == 5, "Thursday")
         .when(F.dayofweek("dateTime") == 6, "Friday")
         .otherwise("Saturday")
    ).groupBy("day_name")
     .agg(
         F.count("*").alias("transaction_count"),
         F.round(F.sum("totalPrice"), 2).alias("total_revenue")
     )
)

---
## Problem 4 — Stay Duration for Bookings

In [ ]:
# Why: datediff(end, start) counts days — for a rental, check_out minus check_in = nights.
result_4 = (
    bookings.withColumn("nights", F.datediff(F.col("check_out"), F.col("check_in")))
            .groupBy("status")
            .agg(
                F.round(F.avg("nights"), 1).alias("avg_nights"),
                F.min("nights").alias("min_nights"),
                F.max("nights").alias("max_nights"),
                F.count("*").alias("booking_count")
            )
)

---
## Problem 5 — Date Formatting and Truncation

In [ ]:
# Why: date_trunc snaps to month boundary; date_format renders it as a readable label.
result_5 = (
    transactions.withColumn("month_label", F.date_format(F.date_trunc("month", "dateTime"), "yyyy-MM"))
                .groupBy("month_label")
                .agg(
                    F.count("*").alias("transaction_count"),
                    F.round(F.sum("totalPrice"), 2).alias("total_revenue")
                )
                .orderBy("month_label")
)